# ISIC Occupation Mapping Pipeline
## LLM-based classification of occupations to ISIC Rev.4 economic activities

This notebook classifies occupations into ISIC sectors using a **prioritised model failover pool**.
When a model hits a rate limit, quota, or produces consecutive failures, the pipeline moves all
remaining occupations to the next available model — the checkpoint preserves all completed work.

| Key | Provider | Free? | Notes |
|---|---|---|---|
| `GROQ_API_KEY` | groq.com | Yes | Primary — 30 RPM, 14 400 RPD per model |
| `OPENROUTER_API_KEY` | openrouter.ai | Yes | Fallback — `:free` models, ~20 RPM |
| `CEREBRAS_API_KEY` | cloud.cerebras.ai | Yes | Very fast LPU; 30 RPM / 60 RPD |
| `TOGETHER_API_KEY` | api.together.xyz | $5 credit | Paid but cheap; most reliable fallback |

**Get keys (all free or near-free):**
- Groq: https://console.groq.com/keys
- OpenRouter: https://openrouter.ai/keys
- Cerebras: https://cloud.cerebras.ai (sign up -> API keys)
- Together AI: https://api.together.xyz/settings/api-keys ($5 signup credit)


In [1]:
import subprocess, sys
pkgs = ['openai', 'groq', 'pandas', 'tqdm', 'boto3', 'openpyxl']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print("All packages installed.")


All packages installed.


In [2]:
import os, json, time, re, logging
from pathlib import Path
from datetime import datetime

import pandas as pd
import boto3
from tqdm import tqdm
from openai import OpenAI
from groq import Groq

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)
print("Imports OK")


Imports OK


### Model failover

The pipeline maintains a live list of available models. For every occupation it tries the first
available model. A `429` / quota error causes that model to be marked unavailable and the next
available model is used immediately. Daily-quota errors disable that model for the rest of the
notebook run; temporary rate limits are cooled down and can become available again.

Set `MODEL_PRIORITY` to control the order. You can use only the providers for which you have keys.


In [3]:
# ============================================================
#              API KEYS  (paste here or set as SageMaker env vars)
# ============================================================
GROQ_API_KEY       = os.getenv('GROQ_API_KEY', '')        # console.groq.com/keys
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY', '')  # openrouter.ai/keys
CEREBRAS_API_KEY   = os.getenv('CEREBRAS_API_KEY', '')    # cloud.cerebras.ai
TOGETHER_API_KEY   = os.getenv('TOGETHER_API_KEY', '')    # api.together.xyz (optional, $5 credit)

# Model priority: pipeline tries models top-to-bottom.
# A model is skipped when rate-limited, quota-exhausted,
# or has CONSEC_FAIL_THRESHOLD consecutive empty/bad-JSON responses.
MODEL_PRIORITY = [
    'groq-llama33-70b',        # Groq llama-3.3-70b-versatile  (128k, free, ~2s/call)
    'groq-llama3-70b',         # Groq llama3-70b-8192           (8k,   free, backup)
    'groq-mixtral-8x7b',       # Groq mixtral-8x7b-32768        (32k,  free, backup)
    'cerebras-llama33-70b',    # Cerebras llama3.3-70b          (fast, free tier)
    'openrouter-llama33',      # OpenRouter llama-3.3-70b:free  (free, 20 RPM)
    'openrouter-qwen25-72b',   # OpenRouter qwen2.5-72b:free    (free, 20 RPM)
    'together-llama33-70b',    # Together AI llama-3.3-70b      ($5 credit, most reliable)
]

# ============================================================
#              AWS / S3 SETTINGS
# ============================================================
USE_S3           = False
S3_BUCKET        = 'YOUR-BUCKET-NAME'
S3_INPUT_PREFIX  = 'isic-mapping/input/'
S3_OUTPUT_PREFIX = 'isic-mapping/output/'

ISIC_CSV_FILENAME        = '/content/ISIC Mapping Phase 01 - Sections (L1).csv'
OCCUPATIONS_CSV_FILENAME = '/content/QS Occupation Taxonomy V2.1- MAY, 2026 - Master - Occ and desc for llms.csv'

# ============================================================
#              PROCESSING / FAILOVER SETTINGS
# ============================================================
CHECKPOINT_EVERY      = 5
MAX_RETRIES           = 2
MAX_NOTE_CHARS        = 600
GROQ_SLEEP            = 2.0   # 30 RPM limit on Groq free tier
OPENROUTER_SLEEP      = 3.5   # ~17 RPM for free OpenRouter models
CEREBRAS_SLEEP        = 2.5   # 30 RPM free tier
TOGETHER_SLEEP        = 1.5   # generous rate limit on paid tier
MAX_TOKENS            = 1200
RATE_LIMIT_COOLDOWN   = 90    # seconds before cooled-down model re-enters pool
CONSEC_FAIL_THRESHOLD = 3     # consecutive empty/bad-JSON responses -> disable model

print('Configuration loaded.')
print(f'  Model priority : {MODEL_PRIORITY}')
print(f'  Using S3       : {USE_S3}')


Configuration loaded.
  Primary model : both
  Using S3      : False


In [4]:
def _load_s3(bucket, key):
    obj = boto3.client('s3').get_object(Bucket=bucket, Key=key)
    return pd.read_csv(obj['Body'])

def _normalise_columns(df, mapping):
    df.columns = df.columns.str.strip()
    rmap = {}
    for std, variants in mapping.items():
        for col in df.columns:
            if col.lower() in [v.lower() for v in variants]:
                rmap[col] = std; break
    return df.rename(columns=rmap)

def load_data():
    if USE_S3:
        isic_df = _load_s3(S3_BUCKET, S3_INPUT_PREFIX + ISIC_CSV_FILENAME)
        occ_df  = _load_s3(S3_BUCKET, S3_INPUT_PREFIX + OCCUPATIONS_CSV_FILENAME)
    else:
        isic_df = pd.read_csv(ISIC_CSV_FILENAME)
        occ_df  = pd.read_csv(OCCUPATIONS_CSV_FILENAME)

    isic_df = _normalise_columns(isic_df, {
        'code':  ['code','isic_code','isic code','Code','Activity Code'],
        'title': ['title','activity title','Title','Activity Title'],
        'explanatory_notes': ['explanatory notes','explanatory_notes',
                               'notes','Explanatory Notes'],
    })
    occ_df = _normalise_columns(occ_df, {
        'occupation_name': ['occupation name','occupation_name',
                             'name','Occupation Name','Occupation'],
        'description':     ['description','occupation description',
                             'job description','Description'],
    })
    logger.info(f'Loaded {len(isic_df)} ISIC, {len(occ_df)} occupations')
    return isic_df, occ_df

isic_df, occupations_df = load_data()

assert 'code'  in isic_df.columns, 'ISIC CSV must have a Code column'
assert 'title' in isic_df.columns, 'ISIC CSV must have a Title column'
assert 'occupation_name' in occupations_df.columns, 'Occupations CSV must have Occupation Name'

print("ISIC cols:", isic_df.columns.tolist())
print("Occ cols: ", occupations_df.columns.tolist())
display(isic_df.head(3))
display(occupations_df.head(3))


ISIC cols: ['code', 'title', 'explanatory_notes']
Occ cols:  ['occupation_name', 'occupation_description']


,code,title,explanatory_notes
0,A,"Agriculture, forestry and fishing",This section includes the exploitation of vege...
1,B,Mining and quarrying,Mining and quarrying include the extraction of...
2,C,Manufacturing,"This section includes the physical, chemical o..."


,occupation_name,occupation_description
0,Plumber,"A Plumber installs, maintains, and repairs wat..."
1,Director of Business Development,A Director of Business Development plans and d...
2,Biofuels Processing Technician,Biofuels Processing Technicians operate and ma...


In [5]:
def build_isic_context(df, max_chars=MAX_NOTE_CHARS):
    blocks = []
    for _, row in df.iterrows():
        c = str(row['code']).strip()
        t = str(row['title']).strip()
        n = str(row.get('explanatory_notes', '')).strip()
        if n.lower() == 'nan': n = ''
        if len(n) > max_chars: n = n[:max_chars] + '...'
        b = f'[{c}] {t}'
        if n: b += f'\n    Note: {n}'
        blocks.append(b)
    return '\n\n'.join(blocks)

ISIC_CONTEXT = build_isic_context(isic_df)
ISIC_CODES   = [str(r['code']).strip() for _, r in isic_df.iterrows()]

est = len(ISIC_CONTEXT) // 4
print(f"ISIC context: {len(ISIC_CONTEXT):,} chars (~{est:,} tokens)")
print(f"ISIC codes  : {len(ISIC_CODES)} entries")
if est > 100_000:
    print('WARNING: context large -- reduce MAX_NOTE_CHARS if hitting limits')
else:
    print('Fits within 128K context window OK')


ISIC context: 12,392 chars (~3,098 tokens)
ISIC codes  : 22 entries
Fits within 128K context window OK


## Prompt Engineering Design

| Technique | Source | Purpose |
|---|---|---|
| Role-based prompting | Brown et al. (2020) GPT-3 | Expert role primes ISIC domain reasoning |
| Chain-of-Thought | Wei et al. (2022) | 4-step decomposition improves ranking by 10-40% |
| Structured JSON schema | Ouyang et al. (2022) | Prevents hallucinated codes, forces completeness |
| Explicit rubric (0/30/70/100) | Zheng et al. (2023) MT-Bench | Makes probability scores consistent across occupations |
| Critical-distinction framing | Turpin et al. (2023) | Separates 'can appear' from 'primarily lives here' |

### One call per occupation (not per occupation x ISIC pair)
Sending all ISIC categories in one prompt enables **cross-ISIC comparative reasoning**: the model
ranks sectors relative to each other rather than scoring each in isolation -- methodologically
superior for ordinal ranking (Ouyang et al., 2022). With 128K context and ~250 tokens per ISIC
entry, up to ~500 ISIC entries fit comfortably.


In [6]:
SYSTEM_PROMPT = (
    'You are an expert economist and occupational analyst specialising in the '
    'International Standard Industrial Classification of All Economic Activities (ISIC) Rev. 4, '
    'published by the United Nations Statistics Division.\n\n'
    'Your task: identify which ISIC economic activity categories are the PRIMARY home '
    'of a given occupation in the labour market.\n\n'
    'CORE DISTINCTION: You are NOT asking whether an occupation CAN exist in a sector. '
    'You are asking WHERE workers with this occupation PRIMARILY create economic value. '
    'A Mining Safety Inspector primarily exists in ISIC B (Mining and Quarrying) even though '
    'safety inspectors appear in many sectors. An Accountant is cross-cutting with no '
    'single dominant sector.\n\n'
    'Output ONLY valid JSON. No text outside the JSON object.'
)

def build_user_prompt(occ_name, occ_desc, isic_context, isic_codes):
    n         = len(isic_codes)
    codes_str = ', '.join(f'"' + c + '"' for c in isic_codes)
    prob_ph   = ',\n    '.join(f'"' + c + '": <0-100>' for c in isic_codes)

    section = lambda title, body: title + '\n' + body

    output_template = (
        '{\n'
        '  "reasoning": {\n'
        '    "occupation_core_function": "<one sentence: what economic function?>",\n'
        '    "primary_sector_logic": "<why Grade A is the correct primary sector>",\n'
        '    "cross_sector_notes": "<is this cross-cutting? what does that mean?>",\n'
        '    "confidence": "high | medium | low"\n'
        '  },\n'
        '  "top_matches": {\n'
        '    "A": { "isic_code": "<EXACT CODE>", "title": "<EXACT TITLE>", "justification": "<evidence>" },\n'
        '    "B": { "isic_code": "<EXACT CODE>", "title": "<EXACT TITLE>", "justification": "<evidence>" },\n'
        '    "C": { "isic_code": "<EXACT CODE>", "title": "<EXACT TITLE>", "justification": "<evidence>" }\n'
        '  },\n'
        '  "probability_scores": {\n'
        f'    {prob_ph}\n'
        '  }\n'
        '}'
    )

    parts = [
        '## OCCUPATION TO CLASSIFY',
        '',
        f'Name        : {occ_name}',
        f'Description : {occ_desc}',
        '',
        '---',
        f'## ALL ISIC ECONOMIC ACTIVITIES ({n} categories)',
        '',
        isic_context,
        '',
        '---',
        '## CLASSIFICATION TASK',
        '',
        'Work through these steps before producing your output.',
        '',
        'STEP 1 -- OCCUPATION CORE ANALYSIS',
        '  What is the fundamental economic function of this occupation?',
        '  Which organisations employ it as part of their PRIMARY (not support) workforce?',
        '',
        'STEP 2 -- GRADE A (Primary sector)',
        '  The ONE ISIC category that is the definitive home of this occupation:',
        '  - Plurality of workers concentrated here',
        '  - This sector core product/service requires this occupation',
        '  - Workers identify with working in this economic activity',
        '',
        'STEP 3 -- GRADE B (Secondary) and GRADE C (Tertiary)',
        '  Second and third strongest sectors.',
        '',
        f'STEP 4 -- PROBABILITY SCORES FOR ALL {n} ISIC CODES',
        '  Assign a score 0-100 to EVERY code including A/B/C:',
        '  A=100, B=85, C=70 by convention. For others:',
        '  90-100 = near-certain, core to sector',
        '  70-89  = high, many workers expected',
        '  50-69  = moderate, meaningful but not primary',
        '  30-49  = low-moderate, occasional workers',
        '  10-29  = low, rare but plausible',
        '   0-9   = very rare or none',
        '',
        'WARNINGS:',
        '- Do NOT award Grade A to generic support functions (HR, IT, Finance, Legal)',
        '  unless explicitly sector-specific (e.g. Petroleum Finance Analyst).',
        '- For cross-cutting occupations (Driver, Cleaner, Accountant, Security Guard)',
        '  note this in reasoning and give distributed moderate scores.',
        '',
        '---',
        '## OUTPUT -- return ONLY this JSON (no text outside the braces):',
        '',
        output_template,
        '',
        f'probability_scores MUST include all {n} codes: {codes_str}',
        'Set A code = 100, B code = 85, C code = 70. All others: 0-89.',
    ]
    return '\n'.join(parts)

# Quick test
_p = build_user_prompt('Mining Engineer',
    'Designs extraction of minerals from underground mines',
    '[B05] Coal Mining\n    Note: Extraction of coal',
    ['B05', 'C24', 'M71'])
print(_p[:600])
print("...")


## OCCUPATION TO CLASSIFY

Name        : Mining Engineer
Description : Designs extraction of minerals from underground mines

---
## ALL ISIC ECONOMIC ACTIVITIES (3 categories)

[B05] Coal Mining
    Note: Extraction of coal

---
## CLASSIFICATION TASK

Work through these steps before producing your output.

STEP 1 -- OCCUPATION CORE ANALYSIS
  What is the fundamental economic function of this occupation?
  Which organisations employ it as part of their PRIMARY (not support) workforce?

STEP 2 -- GRADE A (Primary sector)
  The ONE ISIC category that is the definitive home of this occupation:
 
...


In [7]:
groq_client       = None
openrouter_client = None
cerebras_client   = None
together_client   = None

if GROQ_API_KEY:
    groq_client = Groq(api_key=GROQ_API_KEY)
    logger.info('Groq client ready')
else:
    logger.warning('GROQ_API_KEY not set -- Groq models disabled')

if OPENROUTER_API_KEY:
    openrouter_client = OpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=OPENROUTER_API_KEY,
        default_headers={
            'HTTP-Referer': 'https://github.com/isic-occupation-mapping',
            'X-Title': 'ISIC Occupation Mapping',
        },
    )
    logger.info('OpenRouter client ready')
else:
    logger.warning('OPENROUTER_API_KEY not set -- OpenRouter models disabled')

if CEREBRAS_API_KEY:
    cerebras_client = OpenAI(
        base_url='https://api.cerebras.ai/v1',
        api_key=CEREBRAS_API_KEY,
    )
    logger.info('Cerebras client ready')
else:
    logger.info('CEREBRAS_API_KEY not set -- Cerebras optional')

if TOGETHER_API_KEY:
    together_client = OpenAI(
        base_url='https://api.together.xyz/v1',
        api_key=TOGETHER_API_KEY,
    )
    logger.info('Together AI client ready')
else:
    logger.info('TOGETHER_API_KEY not set -- Together AI optional')

if not any([groq_client, openrouter_client, cerebras_client, together_client]):
    raise RuntimeError('No LLM client initialised. Set at least GROQ_API_KEY or OPENROUTER_API_KEY.')

print(f'Clients: Groq={groq_client is not None} | OpenRouter={openrouter_client is not None}'
      f' | Cerebras={cerebras_client is not None} | Together={together_client is not None}')


Clients: OpenRouter=True | Groq=True


In [8]:
# ============================================================
# MODEL FAILOVER / RATE-LIMIT MANAGER
# ============================================================
class _RL:
    def __init__(self, gap):
        self.gap = gap
        self._last = 0.0
    def wait(self):
        elapsed = time.time() - self._last
        if elapsed < self.gap:
            time.sleep(self.gap - elapsed)
        self._last = time.time()

_rl_groq     = _RL(GROQ_SLEEP)
_rl_or       = _RL(OPENROUTER_SLEEP)
_rl_cerebras = _RL(CEREBRAS_SLEEP)
_rl_together = _RL(TOGETHER_SLEEP)

MODEL_CONFIG = {
    # Groq (fastest, free)
    'groq-llama33-70b': {
        'provider': 'groq',
        'model':    'llama-3.3-70b-versatile',
        'client':   lambda: groq_client,
        'limiter':  _rl_groq,
    },
    'groq-llama3-70b': {
        'provider': 'groq',
        'model':    'llama3-70b-8192',
        'client':   lambda: groq_client,
        'limiter':  _rl_groq,
    },
    'groq-mixtral-8x7b': {
        'provider': 'groq',
        'model':    'mixtral-8x7b-32768',
        'client':   lambda: groq_client,
        'limiter':  _rl_groq,
    },
    # Cerebras (LPU, very fast, free tier)
    'cerebras-llama33-70b': {
        'provider': 'cerebras',
        'model':    'llama3.3-70b',
        'client':   lambda: cerebras_client,
        'limiter':  _rl_cerebras,
    },
    # OpenRouter free tier
    'openrouter-llama33': {
        'provider': 'openrouter',
        'model':    'meta-llama/llama-3.3-70b-instruct:free',
        'client':   lambda: openrouter_client,
        'limiter':  _rl_or,
    },
    'openrouter-qwen25-72b': {
        'provider': 'openrouter',
        'model':    'qwen/qwen-2.5-72b-instruct:free',
        'client':   lambda: openrouter_client,
        'limiter':  _rl_or,
    },
    # Together AI (paid, $5 signup credit, most reliable fallback)
    'together-llama33-70b': {
        'provider': 'together',
        'model':    'meta-llama/Llama-3.3-70B-Instruct-Turbo',
        'client':   lambda: together_client,
        'limiter':  _rl_together,
    },
}

# model_state: available | cooldown | exhausted
# consec_failures: consecutive empty/bad-JSON responses (reset on success)
model_state = {
    name: {'status': 'available', 'disabled_until': 0.0,
           'reason': '', 'consec_failures': 0}
    for name in MODEL_PRIORITY
}


def _is_permanent_error(exc):
    """Non-recoverable errors -- disable model immediately."""
    s = str(exc).lower()
    return any(t in s for t in [
        'decommissioned', 'model not found', 'does not exist',
        'unavailable for free', '404', 'invalid model',
        'model_not_found', 'no such model', 'empty response',
    ])


def _is_quota_error(exc):
    s = str(exc).lower()
    return any(t in s for t in [
        'free-models-per-day', 'requests per day', 'daily quota',
        'daily limit', 'rpd', 'quota exceeded', 'quota limit',
        'remaining requests: 0',
    ])


def _is_rate_limit_error(exc):
    s = str(exc).lower()
    return ('429' in s or 'rate limit' in s or 'rate_limit' in s or
            'too many requests' in s or 'throttl' in s)


def _mark_unavailable(model_name, exc):
    state = model_state[model_name]
    state['consec_failures'] = 0
    if _is_permanent_error(exc) or _is_quota_error(exc):
        state.update(status='exhausted', disabled_until=float('inf'),
                     reason=str(exc)[:120])
        logger.warning(f'[{model_name}] PERMANENT/QUOTA -> disabled: {str(exc)[:80]}')
    else:
        state.update(status='cooldown',
                     disabled_until=time.time() + RATE_LIMIT_COOLDOWN,
                     reason=str(exc)[:120])
        logger.warning(f'[{model_name}] RATE LIMITED -> cooldown {RATE_LIMIT_COOLDOWN}s')


def _record_failure(model_name, reason=''):
    """Count consecutive soft failures (empty / unparseable JSON).
    After CONSEC_FAIL_THRESHOLD, disable model for this run.
    This catches bad model IDs early instead of hammering them 1869 times.
    """
    state = model_state[model_name]
    state['consec_failures'] = state.get('consec_failures', 0) + 1
    state['reason'] = str(reason)[:120]
    count = state['consec_failures']
    if count >= CONSEC_FAIL_THRESHOLD:
        state.update(status='exhausted', disabled_until=float('inf'))
        logger.warning(f'[{model_name}] {count} consecutive soft failures -> DISABLED ({state["reason"]})')
    else:
        logger.info(f'[{model_name}] soft failure {count}/{CONSEC_FAIL_THRESHOLD}: {reason}')


def _record_success(model_name):
    model_state[model_name]['consec_failures'] = 0


def _available_models():
    now = time.time()
    out = []
    for name in MODEL_PRIORITY:
        state = model_state[name]
        if state['status'] == 'exhausted':
            continue
        if state['status'] == 'cooldown':
            if now >= state['disabled_until']:
                state.update(status='available', reason='')
            else:
                continue
        if name not in MODEL_CONFIG:
            continue
        if MODEL_CONFIG[name]['client']() is not None:
            out.append(name)
    return out


def _call_model(model_name, sp, up, temperature=0.0):
    cfg = MODEL_CONFIG[model_name]
    client = cfg['client']()
    if client is None:
        raise RuntimeError(f'{model_name}: client not initialised')
    cfg['limiter'].wait()
    response = client.chat.completions.create(
        model=cfg['model'],
        messages=[
            {'role': 'system', 'content': sp},
            {'role': 'user',   'content': up},
        ],
        temperature=temperature,
        max_tokens=MAX_TOKENS,
    )
    text = response.choices[0].message.content
    if not text or not text.strip():
        raise RuntimeError(f'{model_name} returned an empty response')
    return text


def call_llm(sp, up, max_retries=MAX_RETRIES):
    last_err = None
    for model_name in _available_models():
        for attempt in range(max_retries):
            try:
                text = _call_model(model_name, sp, up, temperature=0.0)
                _record_success(model_name)
                return text, model_name
            except Exception as exc:
                last_err = exc
                if (_is_rate_limit_error(exc) or
                        _is_permanent_error(exc) or
                        _is_quota_error(exc)):
                    _mark_unavailable(model_name, exc)
                    break  # immediate failover to next model
                # Transient error: backoff, then try next model
                wait = (2 ** attempt) * 4
                logger.warning(f'[{model_name}] attempt {attempt+1}/{max_retries}: {exc} (wait {wait}s)')
                if attempt < max_retries - 1:
                    time.sleep(wait)

    available = _available_models()
    exhausted  = [n for n, s in model_state.items() if s['status'] == 'exhausted']
    cooldowns  = [n for n, s in model_state.items() if s['status'] == 'cooldown']
    if exhausted and not available and not cooldowns:
        raise RuntimeError(
            'ALL MODELS EXHAUSTED. Checkpoint is safe. '
            'Resume after quota reset or add more providers. '
            f'Exhausted: {exhausted}'
        )
    raise RuntimeError(
        f'All available models failed. available={available} '
        f'cooldowns={cooldowns} exhausted={exhausted} last={last_err}'
    )


def print_model_status():
    print('\nModel status:')
    for name in MODEL_PRIORITY:
        s = model_state[name]
        cf = s.get('consec_failures', 0)
        print(f'  {name:<26} {s["status"]:<10} fails={cf}  {s["reason"][:70]}')


In [9]:
def _strip_think(t):
    return re.sub(r'<think>[\s\S]*?</think>', '', t, flags=re.IGNORECASE).strip()

def _extract_json(text):
    text = _strip_think(text)
    try: return json.loads(text)
    except: pass
    for p in [r'```json\s*([\s\S]+?)\s*```', r'```([\s\S]+?)```']:
        m = re.search(p, text)
        if m:
            try: return json.loads(m.group(1).strip())
            except: pass
    # Brace-matching scan
    depth = start = None
    depth = 0
    for i, ch in enumerate(text):
        if ch == '{':
            depth += 1
            if start is None: start = i
        elif ch == '}': depth -= 1
        if depth == 0 and start is not None:
            try: return json.loads(text[start:i+1])
            except: start = None; depth = 0
    return None

def _validate(result, isic_codes):
    if not isinstance(result, dict): return None
    result.setdefault('reasoning', {})
    result.setdefault('top_matches', {})
    result.setdefault('probability_scores', {})
    for g, default_score in [('A',100),('B',85),('C',70)]:
        m = result['top_matches'].setdefault(g, {})
        m.setdefault('isic_code', 'UNKNOWN')
        m.setdefault('title', '')
        m.setdefault('justification', '')
        result['probability_scores'][str(m['isic_code']).strip()] = default_score
    for c in isic_codes:
        result['probability_scores'].setdefault(c, 0)
    for c in list(result['probability_scores']):
        try: result['probability_scores'][c] = max(0,min(100,int(float(result['probability_scores'][c]))))
        except: result['probability_scores'][c] = 0
    return result


In [10]:
def classify_occupation(occ_name, occ_desc, isic_context, isic_codes,
                         max_retries=MAX_RETRIES):
    user_p = build_user_prompt(occ_name, occ_desc, isic_context, isic_codes)
    raw = None
    last_model = 'none'

    for attempt in range(max_retries):
        try:
            raw, last_model = call_llm(SYSTEM_PROMPT, user_p, max_retries=1)
            result = _extract_json(raw)
            result = _validate(result, isic_codes)
            if result:
                _record_success(last_model)
                result['_model']   = last_model
                result['_raw_len'] = len(raw)
                return result
            # Response received but JSON unparseable -- count as soft failure
            logger.warning(f'[{last_model}] JSON parse/validation failed; recording failure')
            _record_failure(last_model, 'JSON parse failed')

        except RuntimeError:
            raise   # quota-exhausted signal -- stop pipeline
        except Exception as e:
            logger.error(f'classify error attempt {attempt+1}: {e}')

        if attempt < max_retries - 1:
            time.sleep(2)

    return {
        'reasoning':          {'error': 'failed after all model-pool retries'},
        'top_matches':        {g: {'isic_code': 'ERROR', 'title': '', 'justification': ''} for g in 'ABC'},
        'probability_scores': {c: 0 for c in isic_codes},
        '_model': last_model,
        '_error': str(raw)[:300] if raw else 'no response',
    }


In [11]:
CKPT_DIR  = Path('./checkpoints')
CKPT_FILE = CKPT_DIR / 'isic_mapping_progress.json'
CKPT_DIR.mkdir(exist_ok=True)

def save_checkpoint(results):
    with open(CKPT_FILE, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    logger.info(f'Checkpoint saved ({len(results)} records)')

def load_checkpoint():
    if CKPT_FILE.exists():
        with open(CKPT_FILE) as f: data = json.load(f)
        logger.info(f'Resumed: {len(data)} previously processed')
        return data
    return []


In [ ]:
def run_pipeline(occupations_df, isic_context, isic_codes, resume=True):
    results = load_checkpoint() if resume else []
    # Do not treat prior ERROR rows as completed; remove them so they can be retried.
    results = [r for r in results if r.get('top_matches', {}).get('A', {}).get('isic_code') != 'ERROR']
    done = {r['occupation_name'] for r in results if 'occupation_name' in r and
            r.get('top_matches', {}).get('A', {}).get('isic_code') != 'ERROR'}
    todo = occupations_df[~occupations_df['occupation_name'].isin(done)].copy()

    logger.info(f'Total:{len(occupations_df)}  Done:{len(done)}  Remaining:{len(todo)}')
    print_model_status()

    processed_since_checkpoint = 0
    for i, (_, row) in enumerate(tqdm(todo.iterrows(), total=len(todo), desc='Classifying')):
        name = str(row['occupation_name']).strip()
        desc = str(row.get('description', '')).strip()
        logger.info(f'[{i+1}/{len(todo)}] {name}')

        try:
            res = classify_occupation(name, desc, isic_context, isic_codes)
        except RuntimeError as e:
            # If every provider is exhausted, STOP rather than writing ERROR rows.
            # This keeps the occupation incomplete so the next run can resume it.
            msg = str(e)
            logger.error(msg)
            save_checkpoint(results)
            print_model_status()
            print(f'\nPipeline paused with {len(results)} completed records.')
            print('Re-run this cell after a quota reset/cooldown or after adding another model/provider.')
            return results
        except Exception as e:
            logger.error(f'Unexpected fatal error for {name}: {e}')
            # Keep this occupation incomplete so a later resume can retry it.
            save_checkpoint(results)
            continue

        res['occupation_name'] = name
        res['occupation_description'] = desc
        res['processed_at'] = datetime.now().isoformat()
        results.append(res)
        processed_since_checkpoint += 1

        if processed_since_checkpoint >= CHECKPOINT_EVERY:
            save_checkpoint(results)
            processed_since_checkpoint = 0
            print_model_status()

    save_checkpoint(results)
    logger.info(f'Pipeline complete -- {len(results)} total')
    print_model_status()
    return results

print('=' * 60)
print('STARTING ISIC CLASSIFICATION PIPELINE')
print(f'  Occupations : {len(occupations_df)}')
print(f'  ISIC codes  : {len(ISIC_CODES)}')
print(f'  Models      : {MODEL_PRIORITY}')
print('=' * 60)

ALL_RESULTS = run_pipeline(
    occupations_df=occupations_df,
    isic_context=ISIC_CONTEXT,
    isic_codes=ISIC_CODES,
    resume=True,
)
print(f'Done/Checkpointed: {len(ALL_RESULTS)} occupations classified.')


STARTING ISIC CLASSIFICATION PIPELINE
  Occupations : 1869
  ISIC codes  : 22
  Model       : both


Classifying:   0%|          | 0/1869 [00:00<?, ?it/s]WARNING:__main__:deepseek-r1 attempt 1: Error code: 404 - {'error': {'message': 'This model is unavailable for free. The paid version is available now - use this slug instead: deepseek/deepseek-r1', 'code': 404}, 'user_id': 'user_3Jj6r9XTW1cEPVnXGdGqJt870eN'} (sleep 4s)
ERROR:__main__:Fatal error for Plumber: All LLM calls failed. Last: Error code: 400 - {'error': {'message': 'The model `qwen-qwq-32b` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
Classifying:   0%|          | 1/1869 [00:57<30:00:16, 57.82s/it]WARNING:__main__:deepseek-r1 attempt 1: Error code: 404 - {'error': {'message': 'This model is unavailable for free. The paid version is available now - use this slug instead: deepseek/deepseek-r1', 'code': 404}, 'user_id': 'user_3Jj6r9XTW1cEPVnXG

In [ ]:
def to_wide_df(results, isic_codes):
    rows = []
    for r in results:
        row = {
            'Occupation_Name':        r.get('occupation_name', ''),
            'Occupation_Description': r.get('occupation_description', ''),
            'Model_Used':             r.get('_model', ''),
            'Processed_At':           r.get('processed_at', ''),
            'Confidence':             r.get('reasoning', {}).get('confidence', ''),
            'Core_Function':          r.get('reasoning', {}).get('occupation_core_function', ''),
            'Primary_Sector_Logic':   r.get('reasoning', {}).get('primary_sector_logic', ''),
            'Cross_Sector_Notes':     r.get('reasoning', {}).get('cross_sector_notes', ''),
        }
        for g in ['A', 'B', 'C']:
            m = r.get('top_matches', {}).get(g, {})
            row[f'Grade_{g}_Code']          = m.get('isic_code', '')
            row[f'Grade_{g}_Title']         = m.get('title', '')
            row[f'Grade_{g}_Justification'] = m.get('justification', '')
        ps = r.get('probability_scores', {})
        for c in isic_codes:
            row[f'Prob_{c}'] = ps.get(c, ps.get(str(c), 0))
        rows.append(row)
    return pd.DataFrame(rows)

def to_long_df(results):
    rows = []
    for r in results:
        name = r.get('occupation_name', '')
        gc = {}
        for g in ['A', 'B', 'C']:
            m = r.get('top_matches', {}).get(g, {})
            rows.append({'Occupation_Name': name, 'ISIC_Code': m.get('isic_code',''),
                         'ISIC_Title': m.get('title',''), 'Type': 'Grade', 'Grade': g,
                         'Score': {'A':100,'B':85,'C':70}[g],
                         'Justification': m.get('justification','')})
            gc[m.get('isic_code','')] = True
        for code, score in r.get('probability_scores', {}).items():
            if code not in gc:
                rows.append({'Occupation_Name': name, 'ISIC_Code': code,
                             'ISIC_Title': '', 'Type': 'Probability',
                             'Grade': None, 'Score': score, 'Justification': ''})
    return pd.DataFrame(rows)

results_wide = to_wide_df(ALL_RESULTS, ISIC_CODES)
results_long = to_long_df(ALL_RESULTS)
print(f'Wide: {results_wide.shape}  |  Long: {results_long.shape}')
display(results_wide[['Occupation_Name','Grade_A_Code','Grade_A_Title',
                       'Grade_B_Code','Grade_C_Code','Confidence']].head(5))


In [ ]:
def save_all(results_wide, results_long, all_results):
    ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
    out = Path('./output'); out.mkdir(exist_ok=True)

    wide_csv = out / f'isic_mapping_wide_{ts}.csv'
    long_csv = out / f'isic_mapping_long_{ts}.csv'
    raw_json = out / f'isic_mapping_raw_{ts}.json'
    xlsx     = out / f'isic_mapping_{ts}.xlsx'

    results_wide.to_csv(wide_csv, index=False)
    results_long.to_csv(long_csv, index=False)
    with open(raw_json, 'w') as f: json.dump(all_results, f, indent=2, default=str)

    with pd.ExcelWriter(xlsx, engine='openpyxl') as w:
        results_wide.to_excel(w, sheet_name='Wide_Format', index=False)
        results_long.to_excel(w, sheet_name='Long_Format', index=False)
        pd.DataFrame({'Metric':['Total','Successful','Errors','Timestamp'],
                      'Value': [len(all_results),
                        sum(1 for r in all_results if r.get('top_matches',{}).get('A',{}).get('isic_code','')!='ERROR'),
                        sum(1 for r in all_results if r.get('top_matches',{}).get('A',{}).get('isic_code','')=='ERROR'),
                        ts]}).to_excel(w, sheet_name='Summary', index=False)

    print(f'Saved: {wide_csv}  {long_csv}  {raw_json}  {xlsx}')
    if USE_S3:
        key = f'{S3_OUTPUT_PREFIX}isic_mapping_{ts}.xlsx'
        boto3.client('s3').upload_file(str(xlsx), S3_BUCKET, key)
        print(f'Uploaded to s3://{S3_BUCKET}/{key}')
    return str(xlsx)

excel_path = save_all(results_wide, results_long, ALL_RESULTS)
print('All outputs saved.')


In [ ]:
print('=' * 60)
print('CLASSIFICATION SUMMARY')
print('=' * 60)
total  = len(ALL_RESULTS)
errors = sum(1 for r in ALL_RESULTS
             if r.get('top_matches',{}).get('A',{}).get('isic_code','')=='ERROR')
print(f'Total: {total}  Successful: {total-errors}  Errors: {errors}')

if not results_wide.empty:
    title_map = dict(zip(isic_df['code'].astype(str).str.strip(), isic_df['title']))
    print('\n-- Top 10 Grade-A ISIC Sectors --')
    for code, cnt in results_wide['Grade_A_Code'].value_counts().head(10).items():
        print(f'  [{code}] {title_map.get(str(code).strip(),"Unknown"):<55} -> {cnt} occupations')
    if 'Confidence' in results_wide.columns:
        print('\n-- Confidence Distribution --')
        print(results_wide['Confidence'].value_counts().to_string())
    print('\n-- Models Used --')
    print(results_wide['Model_Used'].value_counts().to_string())


## How to Interpret the Output

### Automatic model failover
Models are tried in `MODEL_PRIORITY` order. A model is skipped when:
- **Rate limit (429)**: temporary cooldown (`RATE_LIMIT_COOLDOWN` s), then re-enters pool
- **Daily/account quota**: disabled for the remainder of the run
- **Permanent error** (404, decommissioned, empty response from bad model ID): disabled immediately
- **Consecutive soft failures**: after `CONSEC_FAIL_THRESHOLD` consecutive empty-response or
  JSON-parse failures, the model is disabled -- catches wrong model IDs within 3 occupations

If **all models are exhausted**, the pipeline stops and saves the checkpoint.
Re-run the cell after a quota reset or after adding a new provider.

### Current model pool (priority order)
| Key | Model ID | Provider | Free? | Context |
|---|---|---|---|---|
| `groq-llama33-70b` | llama-3.3-70b-versatile | Groq | Yes | 128k |
| `groq-llama3-70b` | llama3-70b-8192 | Groq | Yes | 8k |
| `groq-mixtral-8x7b` | mixtral-8x7b-32768 | Groq | Yes | 32k |
| `cerebras-llama33-70b` | llama3.3-70b | Cerebras | Yes (60 RPD) | 8k |
| `openrouter-llama33` | llama-3.3-70b-instruct:free | OpenRouter | Yes | 128k |
| `openrouter-qwen25-72b` | qwen-2.5-72b-instruct:free | OpenRouter | Yes | 128k |
| `together-llama33-70b` | Llama-3.3-70B-Instruct-Turbo | Together AI | $5 credit | 128k |

### Other free providers not yet wired in
- **Mistral AI** (console.mistral.ai): `open-mistral-nemo`, OpenAI-compatible API
- **Nvidia NIM** (build.nvidia.com/nim): `meta/llama-3.1-70b-instruct`, free tier
- **Hugging Face Inference API** (huggingface.co): free but slower
- **Fireworks AI** (fireworks.ai): $1 free credit, fast inference

### Output formats
- **Wide CSV**: one row per occupation; Grade A/B/C columns + `Prob_{code}` for all ISIC codes
- **Long CSV**: one row per (occupation, ISIC code); good for downstream analysis
- **Excel**: both formats as separate sheets + summary statistics
- `Model_Used` column records which provider classified each occupation
